# Lab 5 - Auditory Perception and Signal Processing

In this lab we are going to:


1.   Visualise waveforms and spectrograms
2.   Extract MFCCs
3.   Compare classes
4.   Classify audio with machine learning

First, let's install and import the required libraries




In [ ]:
!pip install numpy
!pip install matplotlib
!pip install librosa
!pip install scikit-learn

import os
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt

import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

Next, we will set a random seed for reproducibility. Also, we'll set some parameters for displaying data later on.

In [ ]:
RNG = 42
random.seed(RNG)
np.random.seed(RNG)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

If you haven't already, download the ESC-50.zip file from the NOW Learning Room. Extract the contents of the zip file so the "ESC-50" directory is in the same place as this Python Notebook.

In [ ]:
!unzip ESC-10.zip # Only run this code if you have uploaded the Zip to Colab or Kaggle!

In [ ]:
ESC10_DIR = Path("ESC-10")

assert ESC10_DIR.exists(), (
    "Could not find ./ESC-10. Make sure you unzipped the folder. Ask your tutor for help if you are not sure how. "
)

classes = sorted([p.name for p in ESC10_DIR.iterdir() if p.is_dir()])
print("Found classes:", classes)


for class_name in classes:
    num_files = len(list((ESC10_DIR / class_name).glob("*.wav")))
    print(f"{class_name} = {num_files} wav files")

# Comparing two files

We will now load two files - helicopter and sea waves. We will then show the wave form of the two files.

**Point of reflection:** do the two files have any similarities and differences? Would you be able to tell which is which?

Once you have run this code, change the files to two classes that you think might be similar. Then try two different classes that you think might be different.

In [ ]:
# Edit these lines of code to specify files
path1 = "ESC-10/helicopter/1-172649-A-40.wav"
path2 = "ESC-10/sea_waves/1-28135-A-11.wav"

# Load audio and print sampling rate
y1, sr1 = librosa.load(path1, sr=None, mono=True)
y2, sr2 = librosa.load(path2, sr=None, mono=True)
print("Sample rate 1:", sr1, "| Duration 1 (s):", round(len(y1)/sr1, 2))
print("Sample rate 2:", sr2, "| Duration 2 (s):", round(len(y2)/sr2, 2))

# Arrange values evenly spaced along the interval (note, this adapts the array to the detected sampling rate)
t1 = np.arange(len(y1)) / sr1
t2 = np.arange(len(y2)) / sr2

plt.figure(figsize=(12, 4))
plt.plot(t1, y1)
plt.title(f"Waveform | {path1}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()
plt.figure(figsize=(12, 4))
plt.plot(t2, y2)
plt.title(f"Waveform | {path2}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

# Spectrograms

Now let's try the spectrograms of the two files you selected.

Note: try this with the two similar wave forms that you found. If you are struggling to find some, revert back to the examples that were provided to you:
> path1 = "ESC-10/helicopter/1-172649-A-40.wav"
> path2 = "ESC-10/sea_waves/1-28135-A-11.wav"

What differences can you see that were not obvious in the time-domain?

In [ ]:
def show_mel_spectrogram(y, sr, title):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(mel_db, sr=sr, hop_length=512, x_axis="time", y_axis="mel")
    plt.colorbar(format="%+2.0f dB")
    plt.title(title)
    plt.tight_layout()
    plt.show()

show_mel_spectrogram(y1, sr1, f"Mel Spectrogram | {path1}")
show_mel_spectrogram(y2, sr2, f"Mel Spectrogram | {path2}")

# Extracting MFCCs

Before performing statistical machine learning, we will extract features to learn from.

For this problem, I have chosen MFCCs.

**Question:** why might MFCCs be a better choice for this dataset than Chroma features?



---



First, let's collect all of the files to form our dataset of raw audio.

To do this, we are simply going to:
1.   Check which folders exist in the ESC-10 folder
2.   Loop through each folder
3.   Collect each audio file and add them to a list of filepaths
4.   As we collect a file, also add the class name to a second list of labels



In [ ]:
classes = sorted([p.name for p in ESC10_DIR.iterdir() if p.is_dir()])
print("Classes:", classes)


filepaths = []
labels = []

for class_name in classes:
    for wav_path in (ESC10_DIR / class_name).glob("*.wav"):
        filepaths.append(wav_path)
        labels.append(class_name)

print("Total clips:", len(filepaths))

Now that we have our raw data ready, we can use librosa to extract the MFCCs.

In [ ]:
def mfcc_features(wav_path, n_mfcc=13):
    # Load audio
    y, sr = librosa.load(wav_path, sr=None, mono=True)

    # Extract MFCCs. Current shape will be (n_mfcc, time_frames)
    # IMPORTANT: this can result in different lengths.
    # Most machine learning models require a fixed length input (shape for all data the same!)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    # To avoid different lengths, we will record the mean and standard deviation as features
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])

print("Let's test to see if it worked correctly")

quick_test = mfcc_features(filepaths[0])
print("File 1 feature length:", len(quick_test))

quick_test = mfcc_features(filepaths[50])
print("File 50 feature length:", len(quick_test))

In [ ]:
X = []
y = []

print("Extracting features...")
print("Note: this can take a while for larger datasets.")
for path, label in zip(filepaths, labels):
    X.append(mfcc_features(path, n_mfcc=13))
    y.append(label)

X = np.vstack(X)
y = np.array(y)

print("X shape:", X.shape)  # (num_samples, num_features)
print("y shape:", y.shape)

print("Features extracted!")

# Machine Learning

Now that our dataset is ready, let's create an 70/30 train/test split and train a machine learning algorithm.

Mini task: in the below code, we define "stratify=y" when generating the train_test_split. Go to the scikit-learn documentation and try to find out more information about stratification. How might this affect our original specification of an 80/20 split?

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training data: ")
print(X_train.shape)

print("Testing data: ")
print(X_test.shape)

Now we will train a small Random Forest with 50 trees on the dataset, and test it on the testing set.

In the below code, the classification report and confusion matrix have been generated.

**Point of reflection:** which classes get confused by the classifier? Why do you think that might be?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

model = RandomForestClassifier(n_estimators=50)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred, labels=classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

plt.figure(figsize=(10, 8))
disp.plot(xticks_rotation=45)
plt.title("Random Forest")
plt.show()


print("Accuracy: ", accuracy_score(y_test, pred))

# Task 1 - Exploring Classifiers and Hyperparameters

In this task you will define a list of classifiers (4 have been provided for you).

Running this code will train and evaluate each classifier, producing a results table. The results are ranked by accuracy.
*Note: we are using macro weighting to treat each class equally.*


---


**Your task:** add machine learning models to the list of models (look for "EDIT THIS BLOCK OF CODE") and explore which approaches are better than others.


Note: remember to also import the models, like I have with:
> from sklearn.ensemble import RandomForestClassifier
> from sklearn.neighbors import KNeighborsClassifier


---

Helpful resources:

A list of all supervised learning models is available at: https://scikit-learn.org/stable/supervised_learning.html

Example python code: https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
import pandas as pd

########## EDIT THIS BLOCK OF CODE
models = [
    ("Random Forest (50 trees)", RandomForestClassifier(n_estimators=50)),
    ("Random Forest (100 trees)", RandomForestClassifier(n_estimators=100)),
    ("KNN (k=5)", KNeighborsClassifier(n_neighbors=5)),
    ("KNN (k=10)", KNeighborsClassifier(n_neighbors=10)),
]
########



results = []

for name, model in models:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="macro", zero_division=0
    )
    results.append({
        "classifier": name,
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    })


results_table = pd.DataFrame(results).sort_values(by="accuracy", ascending=False).reset_index(drop=True)
print(results_table)

# Task 2 - Report and Confusion Matrix

Now that you have explored some models, produce a classification report and confusion matrix to showcase your **best** model.

Note: the code for producing these are available in this notebook.

In [ ]:
# Write your code here!